# Explorer NexaMart sans écrire une ligne de SQL

Ce notebook est **votre porte d'entrée**. Il suppose que vous avez déjà lancé :

```bash
make generate     # ou .\run.ps1 generate sur Windows
make load         # ou .\run.ps1 load
```

Et que `db/nexamart.duckdb` existe donc à la racine du dépôt.

## Objectif

Vous donner **5 visualisations prêtes** pour sentir la forme des
données *avant* de commencer à écrire du SQL. Lancez chaque cellule
avec `Shift+Enter`. Modifiez les requêtes quand une question vous
vient — c'est fait pour ça.

## Prérequis Python (une seule fois)

```bash
pip install -r requirements.txt
```

Installe `duckdb`, `pandas`, `matplotlib`, `jupyter`.

## 1. Se connecter à l'entrepôt (read-only)

On ouvre `db/nexamart.duckdb` en lecture seule. Pas de risque
d'altérer votre entrepôt en jouant dans le notebook.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DB_PATH = Path('..') / 'db' / 'nexamart.duckdb'
assert DB_PATH.exists(), (
    f"Base introuvable: {DB_PATH.resolve()}. "
    "Lancez 'make generate && make load' depuis la racine du depot."
)

con = duckdb.connect(str(DB_PATH), read_only=True)
print(f'Connecte a {DB_PATH.resolve()}')

## 2. Quelles tables avez-vous construites ?

`information_schema.tables` liste toutes les tables du `main` schema.
Les `raw_*` sont les CSV chargés, les `dim_*` sont vos dimensions,
les `fact_*` vos tables de faits. En tout début de trimestre vous
n'aurez que des `raw_*` — c'est normal.

In [ ]:
tables = con.execute("""
    SELECT table_name,
           CASE
               WHEN table_name LIKE 'raw_%'    THEN 'raw'
               WHEN table_name LIKE 'dim_%'    THEN 'dim'
               WHEN table_name LIKE 'fact_%'   THEN 'fact'
               WHEN table_name LIKE 'bridge_%' THEN 'bridge'
               ELSE 'other'
           END AS kind
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY kind, table_name;
""").fetchdf()

counts = []
for _, row in tables.iterrows():
    n = con.execute(f"SELECT COUNT(*) FROM {row['table_name']}").fetchone()[0]
    counts.append({'kind': row['kind'], 'table': row['table_name'], 'rows': n})

pd.DataFrame(counts)

## 3. À quoi ressemble un produit ?

On regarde les 10 premières lignes de `dim_product` pour se familiariser
avec les colonnes disponibles. Remplacez `dim_product` par n'importe
quelle autre table pour explorer.

In [ ]:
# Si dim_product n'existe pas encore (avant S02), on regarde raw_dim_product.
table = 'dim_product'
exists = con.execute(
    "SELECT COUNT(*) FROM information_schema.tables "
    "WHERE table_schema='main' AND table_name=?", [table]
).fetchone()[0]
if not exists:
    table = 'raw_dim_product'
    print(f"dim_product pas encore construite -> on regarde {table}")

con.execute(f"SELECT * FROM {table} LIMIT 10").fetchdf()

## 4. Revenu par catégorie — l'étoile en action

Votre **première vraie question business** : quelle catégorie fait
le plus gros chiffre d'affaires ? On joint `fact_sales` (les mesures)
avec `dim_product` (le contexte) sur `product_key`. C'est la forme
canonique d'une requête en étoile.

Cette cellule saute gracieusement si `fact_sales` n'est pas encore
construite (avant S02).

In [ ]:
try:
    df = con.execute("""
        SELECT p.category,
               ROUND(SUM(f.line_total), 2) AS revenue
        FROM fact_sales  f
        JOIN dim_product p ON p.product_key = f.product_key
        GROUP BY p.category
        ORDER BY revenue DESC;
    """).fetchdf()
    display(df)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(df['category'], df['revenue'])
    ax.invert_yaxis()
    ax.set_xlabel('Revenu ($)')
    ax.set_title('Revenu NexaMart par categorie')
    for i, v in enumerate(df['revenue']):
        ax.text(v, i, f'  {v:,.0f}', va='center')
    plt.tight_layout()
    plt.show()
except duckdb.CatalogException as e:
    print(f"Pas encore disponible (probablement avant S02): {str(e).splitlines()[0]}")

## 5. Tendance mensuelle — le rôle de `dim_date`

Sans `dim_date`, on serait obligé de faire `strftime` partout. Avec,
on agrège par `year_month` en une colonne déjà prête. C'est la raison
d'exister de `dim_date`.

In [ ]:
try:
    df = con.execute("""
        SELECT d.year_month,
               ROUND(SUM(f.line_total), 2) AS revenue,
               COUNT(DISTINCT f.order_number) AS orders
        FROM fact_sales f
        JOIN dim_date   d ON d.date_key = f.order_date
        GROUP BY d.year_month
        ORDER BY d.year_month;
    """).fetchdf()
    display(df)

    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax1.plot(df['year_month'], df['revenue'], marker='o', color='steelblue', label='Revenu')
    ax1.set_xlabel('Mois')
    ax1.set_ylabel('Revenu ($)', color='steelblue')
    ax1.tick_params(axis='x', rotation=45)
    ax2 = ax1.twinx()
    ax2.bar(df['year_month'], df['orders'], alpha=0.25, color='orange', label='Commandes')
    ax2.set_ylabel('Nombre de commandes', color='orange')
    plt.title('Tendance mensuelle : revenu et volume de commandes')
    plt.tight_layout()
    plt.show()
except duckdb.CatalogException as e:
    print(f"Pas encore disponible: {str(e).splitlines()[0]}")

## 6. Top 10 clients — qui fait la loi dans votre entrepôt ?

Jointure avec `dim_customer`. Si vous êtes en SCD Type 2, chaque
client peut apparaître plusieurs fois dans `dim_customer` — assurez-vous
de joindre sur `customer_key` (surrogate), pas `customer_id` (natural),
et d'agréger par `customer_id` pour regrouper les versions.

In [ ]:
try:
    df = con.execute("""
        SELECT c.customer_id,
               MAX(c.full_name) AS full_name,
               ROUND(SUM(f.line_total), 2) AS revenue,
               COUNT(*) AS n_lines
        FROM fact_sales   f
        JOIN dim_customer c ON c.customer_key = f.customer_key
        GROUP BY c.customer_id
        ORDER BY revenue DESC
        LIMIT 10;
    """).fetchdf()
    display(df)
except duckdb.CatalogException as e:
    print(f"Pas encore disponible: {str(e).splitlines()[0]}")

## 7. À vous de jouer

Les cellules ci-dessous sont des **questions business** que vous
pouvez répondre dès que les tables concernées existent. Décommentez,
lancez, modifiez.

Quelques conseils :

- `con.execute("...").fetchdf()` retourne toujours un DataFrame pandas.
- `df.plot(kind='bar', x='...', y='...')` fait un graphe en une ligne.
- `con.execute("DESCRIBE fact_sales").fetchdf()` liste les colonnes
  d'une table.
- Si vous bloquez, voir `docs/TROUBLESHOOTING.md` puis
  `docs/worked-examples/`.

In [ ]:
# Question business 1 : taux de retour par categorie (apres S05)
# df = con.execute("""
#     WITH ventes AS (
#         SELECT p.category, SUM(f.line_total) AS revenue
#         FROM fact_sales f
#         JOIN dim_product p ON p.product_key = f.product_key
#         GROUP BY p.category
#     ),
#     retours AS (
#         SELECT p.category, SUM(r.refund_amount) AS refunds
#         FROM fact_returns r
#         JOIN dim_product p ON p.product_key = r.product_key
#         GROUP BY p.category
#     )
#     SELECT v.category,
#            v.revenue,
#            COALESCE(r.refunds, 0) AS refunds,
#            ROUND(100.0 * COALESCE(r.refunds, 0) / v.revenue, 1) AS return_rate_pct
#     FROM ventes v LEFT JOIN retours r USING (category)
#     ORDER BY return_rate_pct DESC;
# """).fetchdf()
# df

# Question business 2 : revenu par region et par trimestre (apres S02)
# Question business 3 : stock moyen par magasin (apres S04, semi-additive!)
# Question business 4 : delai moyen de livraison (apres S07)
# Question business 5 : conversion apres exposition a une campagne (apres S09)

In [ ]:
con.close()
print('Connexion fermee.')